# Bank deposit classification — full pipeline

## Задачи 

* проанализировать данные о последней маркетинговой кампании, которую проводил банк (задачей было привлечь клиентов для открытия депозита);

* выявить закономерность и найти решающие факторы, повлиявшие на то, что клиент вложил деньги именно в этот банк;

* поднять доходы банка и помочь понять целевую аудиторию, которую необходимо привлекать путём рекламы и различных предложений.

## Данные

**Данные о клиентах банка:**

* age (возраст);

* job (сфера занятости);

* marital (семейное положение);

* education (уровень образования);

* default (имеется ли просроченный кредит);

* housing (имеется ли кредит на жильё);

* loan (имеется ли кредит на личные нужды);

* balance (баланс).

**Данные, связанные с последним контактом в контексте текущей маркетинговой кампании:**

* contact (тип контакта с клиентом);

* month (месяц, в котором был последний контакт);

* day (день, в который был последний контакт);

* duration (продолжительность контакта в секундах).

**Прочие признаки:**

* campaign (количество контактов с этим клиентом в течение текущей кампании);

* pdays (количество пропущенных дней с момента последней маркетинговой кампании до контакта в текущей кампании);

* previous (количество контактов до текущей кампании);

* poutcome (результат прошлой маркетинговой кампании).

**Целевая переменная:** 

* **deposit** (определяет, согласится ли клиент открыть депозит в банке).

## Библиотеки:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing  import LabelEncoder
from sklearn import linear_model 
from sklearn import tree 
from sklearn import ensemble 
from sklearn import metrics 
from sklearn import preprocessing 
from sklearn.model_selection import train_test_split 
from sklearn.feature_selection import SelectKBest, f_classif

## Часть 1. Знакомство с данными, обработка пропусков и выбросов

### Задание 1

In [ ]:
df = pd.read_csv('../data/bank_fin.csv', sep=';')
display(df.head())

In [ ]:
# исследуйте данные на предмет пропусков. Где есть пропущенные значения? Сколько их?

# Проверка пропущенных значений
missing_values = df.isnull().sum()

# Сортируем по количеству пропусков в порядке убывания
missing_sorted = missing_values.sort_values(ascending=False)

# Выводим результат
print("Количество пропущенных значений по признакам:")
print(missing_sorted)

# Ищем признак с максимальным количеством пропусков
max_missing_feature = missing_sorted.index[0]
max_missing_count = missing_sorted.iloc[0]

print(f"\nПризнак с наибольшим количеством пропусков: {max_missing_feature}")
print(f"Количество пропусков: {max_missing_count}")

### Задание 2

In [ ]:
# есть ли в признаке job пропущенные значения? Возможно, они обозначены каким-то специальным словом?

# Проверяем уникальные значения в признаке job
unique_jobs = df['job'].unique()
print("Уникальные значения в столбце 'job':")
print(unique_jobs)
print(f"\nКоличество уникальных значений: {len(unique_jobs)}")

### Задание 3

In [ ]:
# преобразуйте признак balance таким образом, чтобы он корректно считывался, как вещественное число (float)


def as_float(balance):
    try:
        # Если уже число - возвращаем как есть
        if isinstance(balance, (int, float, np.integer, np.floating)):
            return float(balance)
        
        # Если это не строка - пытаемся преобразовать
        if not isinstance(balance, str):
            balance = str(balance)
        
        # Убираем лишние символы
        balance = balance.replace('$', '').replace(' ', '').replace(',', '.')
        
        # Проверяем, что строка не пустая
        if not balance:
            return np.nan
        
        return float(balance)
    
    except (ValueError, TypeError, AttributeError):
        # Если что-то пошло не так - возвращаем NaN
        return np.nan
    

# Применяем функцию
df['balance'] = df['balance'].apply(as_float)

# Проверяем результат
print("Тип данных balance:", df['balance'].dtype)
print("\nПервые 5 значений:")
print(df['balance'].head())
print("\nСтатистика:")
print("Минимум:", df['balance'].min())
print("Максимум:", df['balance'].max())
print("Среднее:", round(df['balance'].mean(), 3))
print("Пропуски:", df['balance'].isnull().sum())

# Проверяем, все ли значения корректно преобразовались
print("\nУникальные типы значений в balance:")
print(df['balance'].apply(type).value_counts())

### Задание 4

In [ ]:
# обработайте пропуски в признаки balance, заменив их на медианные значения по данному признаку

# Вычисляем медиану balance 
median_balance = df['balance'].median()
print(f"Медиана balance: {median_balance:.2f}")

# Заполняем пропуски медианой
df['balance'] = df['balance'].fillna(median_balance)

# Проверяем результат
print(f"Пропуски в balance после заполнения: {df['balance'].isnull().sum()}")

# Вычисляем среднее значение balance
mean_balance = round(df['balance'].mean(), 3)
print(f"\nСреднее значение balance: {mean_balance}")

### Задание 5

In [ ]:
# обработайте пропуски в категориальных признаках: job и education, заменив их на модальные значения

# Заменяем 'unknown' на NaN
df['job'] = df['job'].replace('unknown', np.nan)
df['education'] = df['education'].replace('unknown', np.nan)

# Вычисляем моду job и education 
mode_job = df['job'].mode()[0]
mode_education = df['education'].mode()[0]
print(f"Мода job: {mode_job}")
print(f"Мода education: {mode_education}")

# Заполняем пропуски модой
df['job'] = df['job'].fillna(mode_job)
df['education'] = df['education'].fillna(mode_education)

# Проверяем результат
print(f"Пропуски в job после заполнения: {df['job'].isnull().sum()}")
print(f"Пропуски в education после заполнения: {df['education'].isnull().sum()}")

# Фильтруем клиентов с самой популярной работой и образованием
filtered_df = df[(df['job'] == mode_job) & (df['education'] == mode_education)]

# Вычисляем средний баланс
mean_balance_filtered = round(filtered_df['balance'].mean(), 3)
print(f"\nСредний баланс этих клиентов: {mean_balance_filtered}")

### Задание 6

In [ ]:
# удалите все выбросы для признака balance

# Вычисляем квартили для balance
Q1 = np.percentile(df['balance'], 25)  # 25-й перцентиль (нижний квартиль)
Q3 = np.percentile(df['balance'], 75)  # 75-й перцентиль (верхний квартиль)
IQR = Q3 - Q1  # Межквартильный размах

print(f"Q1 (25-й перцентиль): {Q1:.2f}")
print(f"Q3 (75-й перцентиль): {Q3:.2f}")
print(f"IQR (межквартильный размах): {IQR:.2f}")

# Вычисляем границы по методу Тьюки
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"\nНижняя граница (Q1 - 1.5*IQR): {lower_bound:.2f}")
print(f"Верхняя граница (Q3 + 1.5*IQR): {upper_bound:.2f}")

# Округляем до целых чисел
lower_bound_int = int(round(lower_bound))
upper_bound_int = int(round(upper_bound))

print(f"\nНижняя граница (округлено до целого): {lower_bound_int}")
print(f"Верхняя граница (округлено до целого): {upper_bound_int}")

# Находим выбросы
outliers = df[(df['balance'] < lower_bound) | (df['balance'] > upper_bound)]
print(f"\nКоличество выбросов в balance: {len(outliers)}")
print(f"Процент выбросов: {len(outliers)/len(df)*100:.2f}%")

# Визуализация
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.boxplot(data=df, y='balance')
plt.title('Boxplot balance (с выбросами)')
plt.axhline(y=lower_bound, color='r', linestyle='--', alpha=0.5, label=f'Нижняя граница: {lower_bound_int}')
plt.axhline(y=upper_bound, color='r', linestyle='--', alpha=0.5, label=f'Верхняя граница: {upper_bound_int}')
plt.legend()

plt.subplot(1, 2, 2)
sns.histplot(data=df, x='balance', bins=50)
plt.axvline(x=lower_bound, color='r', linestyle='--', alpha=0.5)
plt.axvline(x=upper_bound, color='r', linestyle='--', alpha=0.5)
plt.title('Распределение balance (с границами выбросов)')

plt.tight_layout()
plt.show()

# Удаляем выбросы
df_no_outliers = df[(df['balance'] >= lower_bound) & (df['balance'] <= upper_bound)].copy()

print(f"\nРазмер данных до удаления выбросов: {len(df)}")
print(f"Размер данных после удаления выбросов: {len(df_no_outliers)}")
print(f"Удалено строк: {len(df) - len(df_no_outliers)}")

# Проверяем, что выбросов не осталось
outliers_after = df_no_outliers[(df_no_outliers['balance'] < lower_bound) | (df_no_outliers['balance'] > upper_bound)]
print(f"\nВыбросов после удаления: {len(outliers_after)}")

In [ ]:
# Сколько объектов осталось после удаления всех выбросов?

df = df_no_outliers.copy()
print(f"Количество объектов после удаления выбросов: {df.shape[0]}")

## Часть 2:  Разведывательный анализ

### Задание 1

In [ ]:
# изучите соотношение классов в ваших данных на предмет несбалансированности, проиллюстрируйте результат

# Анализируем целевую переменную deposit
deposit_counts = df['deposit'].value_counts()
deposit_percentages = df['deposit'].value_counts(normalize=True) * 100

print("Распределение целевой переменной:")
print(deposit_counts)
print("\nРаспределение целевой переменной (в процентах):")
print(deposit_percentages)

# Столбчатая диаграмма
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
sns.countplot(data=df, x='deposit', order=['yes', 'no'])
plt.title('Количество клиентов по решению об открытии депозита')
plt.xlabel('Открыл депозит')
plt.ylabel('Количество клиентов')

### Задания 2 и 3

In [ ]:
df.head()

In [ ]:
# рассчитайте описательные статистики для количественных переменных, проинтерпретируйте результат

# Список числовых признаков
numeric_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

print("ОПИСАТЕЛЬНЫЕ СТАТИСТИКИ ДЛЯ КОЛИЧЕСТВЕННЫХ ПЕРЕМЕННЫХ")
print("=" * 60)

# Основные статистики
desc_stats = df[numeric_cols].describe()
print(desc_stats)

# Дополнительные статистики
print("\nДОПОЛНИТЕЛЬНЫЕ СТАТИСТИКИ:")
print("-" * 40)

for col in numeric_cols:
    print(f"\n{col.upper()}:")
    print(f"  Медиана: {df[col].median():.2f}")
    print(f"  Дисперсия: {df[col].var():.2f}")
    print(f"  Коэффициент вариации: {(df[col].std() / df[col].mean() * 100):.2f}%")
    print(f"  IQR: {df[col].quantile(0.75) - df[col].quantile(0.25):.2f}")

    # Проверка на выбросы (уже удалили для balance, но проверим другие)
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"  Потенциальных выбросов: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)") 

# Визуализация распределений
print("\n" + "=" * 60)
print("ВИЗУАЛИЗАЦИЯ РАСПРЕДЕЛЕНИЙ КОЛИЧЕСТВЕННЫХ ПЕРЕМЕННЫХ")

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    if i < len(axes):
        # Гистограмма с KDE
        sns.histplot(data=df, x=col, kde=True, ax=axes[i], bins=30)
        axes[i].axvline(df[col].mean(), color='red', linestyle='--', label=f'Среднее: {df[col].mean():.1f}')
        axes[i].axvline(df[col].median(), color='green', linestyle='--', label=f'Медиана: {df[col].median():.1f}')
        axes[i].set_title(f'Распределение {col}')
        axes[i].set_xlabel('')
        axes[i].legend(fontsize=8)
        
        # Добавляем boxplot на том же графике (inset)

# Удаляем лишние subplots
for i in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

# Отдельно boxplots для выбросов
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    if i < len(axes):
        sns.boxplot(data=df, y=col, ax=axes[i])
        axes[i].set_title(f'Boxplot {col}')
        axes[i].set_ylabel('')

for i in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

# Максимальный возраст клиента банка
max_age = df['age'].max()
print(f"Максимальный возраст клиента: {max_age} лет")

# Минимальная продолжительность разговора
min_duration = df['duration'].min()
print(f"Минимальная продолжительность разговора: {min_duration} секунд")

### Задания 4 и 5

In [ ]:
# рассчитайте описательные статистики для категориальных переменных, проинтерпретируйте результат
# постройте визуализации, иллюстрирующие результаты

# Список категориальных признаков (кроме target)
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 
                    'loan', 'contact', 'month', 'poutcome']

print("АНАЛИЗ КАТЕГОРИАЛЬНЫХ ПЕРЕМЕННЫХ")
print("=" * 60)

# Создаем визуализации
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    if i < len(axes):
        # Статистика
        value_counts = df[col].value_counts()
        n_unique = df[col].nunique()
        
        print(f"\n{col.upper()}:")
        print(f"  Уникальных значений: {n_unique}")
        print(f"  Распределение:")
        for val, count in value_counts.head().items():
            print(f"    • {val}: {count} ({count/len(df)*100:.1f}%)")
        
        # Визуализация
        if n_unique > 10:  # Если много категорий, покажем топ-10
            top_10 = value_counts.head(10)
            ax = sns.barplot(x=top_10.values, y=top_10.index, ax=axes[i])
            axes[i].set_title(f'{col} (топ-10)')
        else:
            ax = sns.countplot(data=df, y=col, order=value_counts.index, ax=axes[i])
            axes[i].set_title(f'{col}')
            
        axes[i].set_xlabel('Количество')
        axes[i].set_ylabel('')
        
        # Добавляем проценты на график
        total = len(df[col])
        for p in ax.patches:
            width = p.get_width()
            percentage = width/total*100
            ax.text(width + total*0.01, p.get_y() + p.get_height()/2, 
                   f'{width}\n({percentage:.1f}%)', va='center')

# Удаляем лишние subplots
for i in range(len(categorical_cols), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

# ДОПОЛНИТЕЛЬНЫЙ АНАЛИЗ
print("\n" + "=" * 60)
print("ДОПОЛНИТЕЛЬНЫЙ АНАЛИЗ КАТЕГОРИАЛЬНЫХ ПЕРЕМЕННЫХ:")
print("-" * 40)

# 1. Семейное положение
print("\n1. СЕМЕЙНОЕ ПОЛОЖЕНИЕ (marital):")
marital_counts = df['marital'].value_counts()
most_common_marital = marital_counts.idxmax()
print(f"   • Чаще всего: {most_common_marital} ({marital_counts.max()/len(df)*100:.1f}%)")
print(f"   • Распределение: married {marital_counts.get('married', 0)/len(df)*100:.1f}%, "
      f"single {marital_counts.get('single', 0)/len(df)*100:.1f}%, "
      f"divorced {marital_counts.get('divorced', 0)/len(df)*100:.1f}%")

# 2. Образование
print("\n2. ОБРАЗОВАНИЕ (education):")
edu_counts = df['education'].value_counts()
most_common_edu = edu_counts.idxmax()
print(f"   • Чаще всего: {most_common_edu} ({edu_counts.max()/len(df)*100:.1f}%)")
print(f"   • Уровни: secondary {edu_counts.get('secondary', 0)/len(df)*100:.1f}%, "
      f"tertiary {edu_counts.get('tertiary', 0)/len(df)*100:.1f}%, "
      f"primary {edu_counts.get('primary', 0)/len(df)*100:.1f}%")

# 3. Результат прошлой кампании
print("\n3. РЕЗУЛЬТАТ ПРОШЛОЙ КАМПАНИИ (poutcome):")
poutcome_counts = df['poutcome'].value_counts()
print(f"   • Успех прошлой кампании: {poutcome_counts.get('success', 0)} клиентов")
print(f"   • Неизвестно: {poutcome_counts.get('unknown', 0)} ({poutcome_counts.get('unknown', 0)/len(df)*100:.1f}%)")

# 4. Контакт
print("\n4. ТИП КОНТАКТА (contact):")
contact_counts = df['contact'].value_counts()
print(f"   • Cellular: {contact_counts.get('cellular', 0)} ({contact_counts.get('cellular', 0)/len(df)*100:.1f}%)")
print(f"   • Telephone: {contact_counts.get('telephone', 0)} ({contact_counts.get('telephone', 0)/len(df)*100:.1f}%)")

# 5. Кредиты
print("\n5. НАЛИЧИЕ КРЕДИТОВ:")
print(f"   • Ипотека (housing): Да - {df['housing'].value_counts().get('yes', 0)} ({df['housing'].value_counts().get('yes', 0)/len(df)*100:.1f}%)")
print(f"   • Потребительский кредит (loan): Да - {df['loan'].value_counts().get('yes', 0)} ({df['loan'].value_counts().get('yes', 0)/len(df)*100:.1f}%)")
print(f"   • Просроченный кредит (default): Да - {df['default'].value_counts().get('yes', 0)} ({df['default'].value_counts().get('yes', 0)/len(df)*100:.1f}%)")

# Проверка данных по месяцам более подробно
print("\n" + "=" * 60)
print("ДЕТАЛЬНЫЙ АНАЛИЗ МЕСЯЦЕВ КАМПАНИИ:")
print("-" * 40)

month_counts = df['month'].value_counts().sort_index()
print("\nКоличество контактов по месяцам:")
for month, count in month_counts.items():
    print(f"  {month}: {count} контактов")

# Визуализация по месяцам
plt.figure(figsize=(10, 6))
month_order = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
# Только те месяцы, которые есть в данных
available_months = [m for m in month_order if m in df['month'].unique()]
sns.countplot(data=df, x='month', order=available_months)
plt.title('Активность маркетинговой кампании по месяцам')
plt.xlabel('Месяц')
plt.ylabel('Количество контактов')
plt.xticks(rotation=45)
plt.show()

# Сколько месяцев проводилась кампания?
unique_months = df['month'].nunique()
all_months = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 
              'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
present_months = df['month'].unique()
missing_months = [m for m in all_months if m not in present_months]

print(f"Маркетинговая кампания проводилась в {unique_months} месяцах")
print(f"   Месяцы кампании: {sorted(present_months)}")
if missing_months:
    print(f"   НЕ проводилась в: {missing_months}")

# Сколько сфер занятости?
unique_jobs = df['job'].nunique()
print(f"\nСфер занятости представлено: {unique_jobs}")
print(f"   Самые популярные:")
for job, count in df['job'].value_counts().head(3).items():
    print(f"   • {job}: {count} клиентов ({count/len(df)*100:.1f}%)")

### Задание 6

In [ ]:
# Узнайте, для какого статуса предыдущей маркетинговой кампании успех в текущей превалирует над количеством неудач.

# Анализируем связь между результатом прошлой и текущей кампаний
print("СВЯЗЬ МЕЖДУ РЕЗУЛЬТАТАМИ ПРОШЛОЙ И ТЕКУЩЕЙ КАМПАНИЙ")
print("=" * 60)

# Создаем кросс-таблицу
cross_tab = pd.crosstab(df['poutcome'], df['deposit'], normalize='index') * 100
cross_tab_counts = pd.crosstab(df['poutcome'], df['deposit'])

print("\nРаспределение текущего результата по результатам прошлой кампании (%):")
print(cross_tab.round(1))
print("\nАбсолютные значения:")
print(cross_tab_counts)

# Вычисляем соотношение успехов к неудачам для каждого статуса прошлой кампании
print("\n" + "=" * 60)
print("СООТНОШЕНИЕ УСПЕХОВ К НЕУДАЧАМ:")
print("-" * 40)

success_ratios = {}
for status in cross_tab_counts.index:
    successes = cross_tab_counts.loc[status, 'yes']
    failures = cross_tab_counts.loc[status, 'no']
    
    if failures > 0:
        ratio = successes / failures
    else:
        ratio = float('inf')  # если нет неудач
    
    success_ratios[status] = ratio
    
    print(f"\n{status.upper()}:")
    print(f"  Успехов (deposit='yes'): {successes}")
    print(f"  Неудач (deposit='no'): {failures}")
    print(f"  Соотношение успехов/неудач: {ratio:.2f}")
    
    if ratio > 1:
        print(f"  ✓ Успехов БОЛЬШЕ, чем неудач (в {ratio:.1f} раза)")
    elif ratio == 1:
        print(f"  → Успехов и неудач ПОРОВНУ")
    else:
        print(f"  ✗ Успехов МЕНЬШЕ, чем неудач")

# Находим статус, где успехов намного больше, чем неудач
print("\n" + "=" * 60)
print("АНАЛИЗ: ГДЕ УСПЕХОВ НАМНОГО БОЛЬШЕ, ЧЕМ НЕУДАЧ?")
print("-" * 40)

# Определяем "намного больше" - например, в 2 раза или больше
threshold = 2.0
candidates = []

for status, ratio in success_ratios.items():
    if ratio >= threshold:
        candidates.append((status, ratio))
        print(f"  {status}: соотношение = {ratio:.2f} (успехов в {ratio:.1f} раза больше)")

if candidates:
    # Находим максимальное соотношение
    best_status, best_ratio = max(candidates, key=lambda x: x[1])
    print(f"\n✓ Наибольшее преимущество у статуса: '{best_status}'")
    print(f"  Соотношение успехов/неудач: {best_ratio:.2f}")
    print(f"  Это значит: успехов в {best_ratio:.1f} раза больше, чем неудач")
else:
    print("\n✗ Нет статусов, где успехов в 2+ раза больше, чем неудач")
    # Ищем просто наибольшее соотношение (даже если < 2)
    best_status = max(success_ratios.items(), key=lambda x: x[1])
    print(f"  Наибольшее соотношение у '{best_status[0]}': {best_status[1]:.2f}")

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. Столбчатая диаграмма с группировкой
ax1 = axes[0]
cross_tab_counts.plot(kind='bar', ax=ax1)
ax1.set_title('Результаты текущей кампании по статусам прошлой')
ax1.set_xlabel('Статус предыдущей кампании')
ax1.set_ylabel('Количество клиентов')
ax1.legend(title='Текущий депозит')
ax1.tick_params(axis='x', rotation=45)

# Добавляем проценты на первый график
for i, (idx, row) in enumerate(cross_tab_counts.iterrows()):
    total = row.sum()
    yes_pct = row['yes'] / total * 100
    no_pct = row['no'] / total * 100
    ax1.text(i - 0.2, row['yes'] + total*0.02, f'{yes_pct:.0f}%', ha='center')
    ax1.text(i + 0.2, row['no'] + total*0.02, f'{no_pct:.0f}%', ha='center')

# 2. Heatmap с процентами
ax2 = axes[1]
sns.heatmap(cross_tab, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax2)
ax2.set_title('Доля успехов/неудач по статусам прошлой кампании (%)')
ax2.set_xlabel('Текущий депозит')
ax2.set_ylabel('Статус предыдущей кампании')

plt.tight_layout()
plt.show()

# Дополнительный анализ: абсолютные числа
print("\n" + "=" * 60)
print("ДОПОЛНИТЕЛЬНЫЙ АНАЛИЗ:")
print("-" * 40)

# Смотрим на 'success' статус отдельно
if 'success' in cross_tab_counts.index:
    success_stats = cross_tab_counts.loc['success']
    success_total = success_stats.sum()
    success_yes = success_stats['yes']
    success_yes_pct = success_yes / success_total * 100
    
    print(f"\nКлиенты с УСПЕШНОЙ прошлой кампанией ('success'):")
    print(f"  • Всего: {success_total} клиентов")
    print(f"  • Открыли депозит сейчас: {success_yes} ({success_yes_pct:.1f}%)")
    print(f"  • НЕ открыли депозит: {success_stats['no']} ({100-success_yes_pct:.1f}%)")

# Сравниваем с другими статусами
print(f"\nСравнение всех статусов по % успеха в текущей кампании:")
for status in cross_tab_counts.index:
    total = cross_tab_counts.loc[status].sum()
    yes_count = cross_tab_counts.loc[status, 'yes']
    yes_pct = yes_count / total * 100
    print(f"  {status}: {yes_pct:.1f}% успеха ({yes_count}/{total})")

### Задание 7

In [ ]:
df.head()

In [ ]:
# узнайте, в каком месяце чаще всего отказывались от предложения открыть депозит

df_no_deposit = df[df['deposit'] == 'no'].groupby('month')['deposit'].count()
df_total_deposit = df.groupby('month')['deposit'].count()
# Считаем процент
failure_percentage = (df_no_deposit / df_total_deposit * 100).sort_values(ascending=False)
display(failure_percentage)

# Найти месяц с максимальным значением
max_month = failure_percentage.idxmax()
max_value = failure_percentage.max()

print(f"Чаще всего отказывались в месяце: {max_month}")
print(f"Процент отказов: {round(max_value, 1)}")

### Задание 8

In [ ]:
# создайте возрастные группы и определите, в каких группах более склонны открывать депозит, чем отказываться от предложения

# 1. Создаем возрастные группы
def get_age_group(age):
    if age < 30: return '<30'
    elif age < 40: return '30-40'
    elif age < 50: return '40-50'
    elif age < 60: return '50-60'
    else: return '60+'

df['age_group'] = df['age'].apply(get_age_group)

# 2. Анализ
age_deposit = pd.crosstab(df['age_group'], df['deposit'], normalize='index') * 100
age_counts = pd.crosstab(df['age_group'], df['deposit'])

print("Процент открывших депозит по возрастным группам:")
print(age_deposit.round(1))

print("\nГде больше склонны открывать депозит?")
for group in age_deposit.index:
    yes_pct = age_deposit.loc[group, 'yes']
    no_pct = age_deposit.loc[group, 'no']
    if yes_pct > no_pct:
        print(f"  {group}: ДА ({yes_pct:.1f}% > {no_pct:.1f}%)")

# 3. Визуализация 
plt.figure(figsize=(10, 6))
age_deposit.plot(kind='bar')
plt.title('Процент открытий депозита по возрастным группам')
plt.xlabel('Возрастная группа')
plt.ylabel('Процент (%)')
plt.legend(title='Депозит')
plt.show()

### Задания 9 и 10

In [ ]:
# постройте визуализации для открывших и неоткрывших депозит в зависимости от семейного статуса

# Визуализация для семейного статуса
marital_cross = pd.crosstab(df['marital'], df['deposit'])

plt.figure(figsize=(10, 6))
marital_cross.plot(kind='bar', color=['#ff6b6b', '#4ecdc4'])
plt.title('Открытие депозита по семейному положению')
plt.xlabel('Семейное положение')
plt.ylabel('Количество клиентов')
plt.legend(title='Депозит', labels=['Нет', 'Да'])
plt.xticks(rotation=0)

# Добавляем числа на столбцы
for container in plt.gca().containers:
    plt.gca().bar_label(container, fontsize=10)

plt.show()

In [ ]:
# постройте визуализации для открывших и неоткрывших депозит в зависимости от образования

# Визуализация для образования
edu_cross = pd.crosstab(df['education'], df['deposit'])

plt.figure(figsize=(10, 6))
edu_cross.plot(kind='bar', color=['#ff6b6b', '#4ecdc4'])
plt.title('Открытие депозита по уровню образования')
plt.xlabel('Образование')
plt.ylabel('Количество клиентов')
plt.legend(title='Депозит', labels=['Нет', 'Да'])
plt.xticks(rotation=0)

# Добавляем числа на столбцы
for container in plt.gca().containers:
    plt.gca().bar_label(container, fontsize=10)

plt.show()

In [ ]:
# постройте визуализации для открывших и неоткрывших депозит в зависимости от вида профессиональной занятости

# Визуализация для профессиональной занятости
job_cross = pd.crosstab(df['job'], df['deposit'])

plt.figure(figsize=(12, 7))
job_cross.plot(kind='bar', color=['#ff6b6b', '#4ecdc4'])
plt.title('Открытие депозита по сфере занятости')
plt.xlabel('Профессия')
plt.ylabel('Количество клиентов')
plt.legend(title='Депозит', labels=['Нет', 'Да'])
plt.xticks(rotation=45, ha='right')

# Добавляем числа на столбцы
for container in plt.gca().containers:
    plt.gca().bar_label(container, fontsize=8)

plt.tight_layout()
plt.show()

### Задание 11

In [ ]:
# постройте сводную таблицу, чтобы определить люди с каким образованием и семейным статусом наиболее многочисленны
#(если рассматривать тех, кто открыл депозит)

# Разделяем данные
df_yes = df[df['deposit'] == 'yes']  # открыли депозит
df_no = df[df['deposit'] == 'no']    # не открыли депозит

# Создаем сводные таблицы
pivot_yes = pd.crosstab(df_yes['education'], df_yes['marital'])
pivot_no = pd.crosstab(df_no['education'], df_no['marital'])

# Тепловые карты
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. Открыли депозит
sns.heatmap(pivot_yes, annot=True, fmt='d', cmap='Greens', ax=axes[0])
axes[0].set_title('Открыли депозит: Образование × Семейное положение')
axes[0].set_xlabel('Семейное положение')
axes[0].set_ylabel('Образование')

# 2. Не открыли депозит
sns.heatmap(pivot_no, annot=True, fmt='d', cmap='Reds', ax=axes[1])
axes[1].set_title('Не открыли депозит: Образование × Семейное положение')
axes[1].set_xlabel('Семейное положение')
axes[1].set_ylabel('Образование')

plt.tight_layout()
plt.show()

# Анализ результатов
print("АНАЛИЗ РЕЗУЛЬТАТОВ:")
print("="*60)

print("\n1. Для ОТКРЫВШИХ депозит:")
max_yes = pivot_yes.max().max()
max_yes_idx = pivot_yes.stack().idxmax()
print(f"   Самая частая группа: {max_yes_idx[0]} образование, {max_yes_idx[1]} статус")
print(f"   Количество: {max_yes} клиентов")

print("\n2. Для НЕ ОТКРЫВШИХ депозит:")
max_no = pivot_no.max().max()
max_no_idx = pivot_no.stack().idxmax()
print(f"   Самая частая группа: {max_no_idx[0]} образование, {max_no_idx[1]} статус")
print(f"   Количество: {max_no} клиентов")

print("\n3. Сравнение:")
print(f"   Одинокие с высшим образованием (single + tertiary):")
print(f"     - Открыли: {pivot_yes.loc['tertiary', 'single'] if 'tertiary' in pivot_yes.index and 'single' in pivot_yes.columns else 0}")
print(f"     - Не открыли: {pivot_no.loc['tertiary', 'single'] if 'tertiary' in pivot_no.index and 'single' in pivot_no.columns else 0}")

print(f"\n   Разведённые с начальным образованием (divorced + primary):")
print(f"     - Открыли: {pivot_yes.loc['primary', 'divorced'] if 'primary' in pivot_yes.index and 'divorced' in pivot_yes.columns else 0}")
print(f"     - Не открыли: {pivot_no.loc['primary', 'divorced'] if 'primary' in pivot_no.index and 'divorced' in pivot_no.columns else 0}")

# Процентное соотношение для наглядности
print("\n" + "="*60)
print("ПРОЦЕНТНОЕ РАСПРЕДЕЛЕНИЕ:")
print("="*60)

# Процентные тепловые карты
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Нормализуем по общему количеству
pivot_yes_pct = (pivot_yes / len(df_yes) * 100).round(1)
pivot_no_pct = (pivot_no / len(df_no) * 100).round(1)

sns.heatmap(pivot_yes_pct, annot=True, fmt='.1f', cmap='Greens', ax=axes[0])
axes[0].set_title('Открыли депозит (% от всех открывших)')
axes[0].set_xlabel('Семейное положение')
axes[0].set_ylabel('Образование')

sns.heatmap(pivot_no_pct, annot=True, fmt='.1f', cmap='Reds', ax=axes[1])
axes[1].set_title('Не открыли депозит (% от всех не открывших)')
axes[1].set_xlabel('Семейное положение')
axes[1].set_ylabel('Образование')

plt.tight_layout()
plt.show()

## Часть 3: преобразование данных

### Задание 1

In [ ]:
# преобразуйте уровни образования

# Создаем копию LabelEncoder
le = LabelEncoder()

# Преобразуем education
df['education_encoded'] = le.fit_transform(df['education'])

# Проверяем результат
print("Преобразование education:")
print(pd.DataFrame({
    'education': df['education'].unique(),
    'encoded': le.transform(df['education'].unique())
}))

print("\nПервые 10 значений:")
print(df[['education', 'education_encoded']].head(10))

# Находим сумму значений
sum_education = df['education_encoded'].sum()
print(f"\nСумма значений education_encoded: {sum_education}")

### Задания 2 и 3

In [ ]:
# преобразуйте бинарные переменные в представление из нулей и единиц

# Преобразуем age_group в порядковый признак
age_group_mapping = {
    '<30': 0,
    '30-40': 1,
    '40-50': 2,
    '50-60': 3,
    '60+': 4
}

df['age_group_encoded'] = df['age_group'].map(age_group_mapping)

print("Кодирование age_group:")
for key, value in age_group_mapping.items():
    count = len(df[df['age_group'] == key])
    print(f"  {key} → {value} (количество: {count})")

print(f"\nПервые 10 значений:")
print(df[['age_group', 'age_group_encoded']].head(10))

# Преобразуем deposit в 0 и 1
deposit_mapping = {'no': 0, 'yes': 1}
df['deposit_binary'] = df['deposit'].map(deposit_mapping)

print("Кодирование deposit:")
print(f"  no → 0")
print(f"  yes → 1")

print(f"\nРаспределение:")
print(f"  0 (нет): {len(df[df['deposit_binary'] == 0])}")
print(f"  1 (да): {len(df[df['deposit_binary'] == 1])}")

# Вычисляем стандартное отклонение
std_deposit = df['deposit_binary'].std()
std_rounded = round(std_deposit, 3)

print(f"\nСтандартное отклонение deposit_binary: {std_deposit}")
print(f"Стандартное отклонение (округлено до 3 знаков): {std_rounded}")

# Бинарные переменные для преобразования
binary_cols = ['default', 'housing', 'loan']

print("ПРЕОБРАЗОВАНИЕ БИНАРНЫХ ПЕРЕМЕННЫХ:")
print("="*50)

# Преобразуем каждую переменную
sum_of_means = 0

for col in binary_cols:
    # Преобразуем yes→1, no→0
    df[f'{col}_binary'] = df[col].map({'yes': 1, 'no': 0})
    
    # Вычисляем статистику
    mean_val = df[f'{col}_binary'].mean()
    count_yes = df[f'{col}_binary'].sum()  # сумма = количество yes (1)
    count_no = len(df) - count_yes
    
    print(f"\n{col.upper()}:")
    print(f"  yes (1): {count_yes} клиентов ({mean_val*100:.1f}%)")
    print(f"  no (0): {count_no} клиентов ({(1-mean_val)*100:.1f}%)")
    print(f"  Среднее значение: {mean_val:.3f}")
    
    sum_of_means += mean_val

print("\n" + "="*50)
print(f"Сумма средних значений: {sum_of_means:.3f}")
print(f"Сумма средних (округлено до 3 знаков): {round(sum_of_means, 3)}")

# Проверка преобразования
print("\nПервые 5 строк после преобразования:")
for col in binary_cols:
    print(f"{col}: {df[col].iloc[0]} → {col}_binary: {df[f'{col}_binary'].iloc[0]}")

### Задание 4

In [ ]:
# создайте дамми-переменные

# Максимально простой вариант
df = pd.get_dummies(df, columns=['job', 'marital', 'contact', 'month', 'poutcome'])

# Считаем
all_cols = df.columns.tolist()
targets = ['deposit', 'deposit_binary']
features = [col for col in all_cols if col not in targets]

print(f"Ответ: {len(features)}") 

### Задания 5 и 6

In [ ]:
# постройте корреляционную матрицу и оцените данные на предмет наличия мультиколлинеарности

# 1. Готовим данные для корреляции
# Берем только числовые признаки и целевую переменную deposit_binary
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Убедимся что deposit_binary есть в списке
if 'deposit_binary' not in numerical_cols and 'deposit_binary' in df.columns:
    numerical_cols.append('deposit_binary')

# Создаем DataFrame только с числовыми признаками
corr_df = df[numerical_cols]

# 2. Строим корреляционную матрицу
correlation_matrix = corr_df.corr()

print("РАЗМЕР КОРРЕЛЯЦИОННОЙ МАТРИЦЫ:")
print(f"Количество признаков: {len(correlation_matrix)}")
print(f"Размер матрицы: {correlation_matrix.shape}")

# 3. Тепловая карта корреляций
plt.figure(figsize=(16, 12))
sns.heatmap(correlation_matrix, 
            annot=True, 
            fmt='.2f', 
            cmap='coolwarm',
            center=0,
            square=True,
            linewidths=0.5,
            cbar_kws={"shrink": 0.8})
plt.title('Матрица корреляций', fontsize=16, pad=20)
plt.tight_layout()
plt.show()

# 4. Корреляция с целевой переменной
print("\nКОРРЕЛЯЦИЯ С ЦЕЛЕВОЙ ПЕРЕМЕННОЙ (deposit_binary):")
target_corr = correlation_matrix['deposit_binary'].sort_values(ascending=False)

print("Топ-10 наибольших корреляций с deposit_binary:")
for i, (feature, corr) in enumerate(target_corr.head(11).items()):  # 11 т.к. первая - deposit_binary с самой собой
    if feature != 'deposit_binary':
        print(f"{i:2d}. {feature:30} : {corr:+.3f}")

print("\nТоп-10 наименьших корреляций с deposit_binary:")
for i, (feature, corr) in enumerate(target_corr.tail(10).items()):
    print(f"{i:2d}. {feature:30} : {corr:+.3f}")

# 5. Столбчатая диаграмма корреляций
plt.figure(figsize=(14, 8))
# Берем корреляции без deposit_binary с самой собой
target_corr_no_self = target_corr.drop('deposit_binary', errors='ignore')
target_corr_sorted = target_corr_no_self.sort_values(ascending=False)

# Создаем цветовую схему: положительные - зеленые, отрицательные - красные
colors = ['green' if x > 0 else 'red' for x in target_corr_sorted]

bars = plt.barh(target_corr_sorted.index, target_corr_sorted.values, color=colors)
plt.title('Корреляция признаков с целевой переменной (deposit_binary)', fontsize=14)
plt.xlabel('Коэффициент корреляции')
plt.ylabel('Признаки')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)

# Добавляем значения на столбцы
for bar in bars:
    width = bar.get_width()
    plt.text(width + (0.01 if width >= 0 else -0.03), 
             bar.get_y() + bar.get_height()/2,
             f'{width:.3f}', 
             ha='left' if width >= 0 else 'right',
             va='center',
             fontsize=9)

plt.gca().invert_yaxis()  # чтобы наибольшие корреляции были сверху
plt.tight_layout()
plt.show()

# 6. Анализ мультиколлинеарности
print("\n" + "="*60)
print("АНАЛИЗ МУЛЬТИКОЛЛИНЕАРНОСТИ:")
print("="*60)

# Ищем высокие корреляции между признаками (исключая deposit_binary)
high_corr_pairs = []
threshold = 0.7  # порог для сильной корреляции

for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        col1 = correlation_matrix.columns[i]
        col2 = correlation_matrix.columns[j]
        corr_value = abs(correlation_matrix.iloc[i, j])
        
        if corr_value > threshold and col1 != 'deposit_binary' and col2 != 'deposit_binary':
            high_corr_pairs.append((col1, col2, corr_value))

if high_corr_pairs:
    print(f"Найдены сильные корреляции (>{threshold}):")
    for col1, col2, corr in sorted(high_corr_pairs, key=lambda x: x[2], reverse=True):
        print(f"  {col1:30} ↔ {col2:30} : {corr:.3f}")
else:
    print(f"Сильных корреляций (>{threshold}) между признаками не обнаружено")

# 7. VIF анализ (более точный метод проверки мультиколлинеарности)
try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    
    # Берем только признаки (без целевой)
    features_for_vif = [col for col in numerical_cols if col != 'deposit_binary']
    X_for_vif = df[features_for_vif]
    
    # Рассчитываем VIF
    vif_data = pd.DataFrame()
    vif_data["feature"] = X_for_vif.columns
    vif_data["VIF"] = [variance_inflation_factor(X_for_vif.values, i) 
                       for i in range(len(X_for_vif.columns))]
    
    print("\nVIF (Фактор инфляции дисперсии):")
    print("VIF > 10 указывает на серьезную мультиколлинеарность")
    print(vif_data.sort_values("VIF", ascending=False).head(15))
    
except ImportError:
    print("\nДля VIF анализа требуется установить statsmodels: pip install statsmodels")

### Задания 7 и 8

In [ ]:
# Определяем предикторы и целевую переменную
# Используем deposit_binary как целевую (0/1)
X = df.drop(['deposit', 'deposit_binary', 'deposit_encoded'], axis=1, errors='ignore')
y = df['deposit_binary']  # или можно df['deposit_encoded']

# Разделяем на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    stratify=y, 
    random_state=42, 
    test_size=0.33
)

print("РАЗМЕРЫ ВЫБОРОК:")
print("="*40)
print(f"Исходный размер данных: {len(df)}")
print(f"Размер X (признаки): {X.shape}")
print(f"Размер y (целевая): {y.shape}")
print(f"\nОбучающая выборка (X_train): {X_train.shape}")
print(f"Обучающая выборка (y_train): {y_train.shape}")
print(f"Тестовая выборка (X_test): {X_test.shape}")
print(f"Тестовая выборка (y_test): {y_test.shape}")

# Размер тестовой выборки (количество строк)
test_size_count = X_test.shape[0]
print(f"\nРазмер тестовой выборки (количество строк): {test_size_count}")

# Проверяем распределение целевой переменной
print("\nРАСПРЕДЕЛЕНИЕ ЦЕЛЕВОЙ ПЕРЕМЕННОЙ:")
print("="*40)
print(f"Обучающая выборка (train):")
print(f"  0 (нет): {sum(y_train == 0)} ({sum(y_train == 0)/len(y_train)*100:.1f}%)")
print(f"  1 (да):  {sum(y_train == 1)} ({sum(y_train == 1)/len(y_train)*100:.1f}%)")

print(f"\nТестовая выборка (test):")
print(f"  0 (нет): {sum(y_test == 0)} ({sum(y_test == 0)/len(y_test)*100:.1f}%)")
print(f"  1 (да):  {sum(y_test == 1)} ({sum(y_test == 1)/len(y_test)*100:.1f}%)")

In [ ]:
# рассчитайте необходимые показатели

# Вычисляем среднее значение целевой переменной на тестовой выборке
mean_test = y_test.mean()
mean_test_rounded = round(mean_test, 2)

print(f"Среднее значение целевой переменной на тестовой выборке: {mean_test:.4f}")
print(f"Среднее значение (округлено до 2 знаков): {mean_test_rounded}")

# Объяснение:
# Для бинарной переменной (0/1) среднее = доля единиц (доля "ДА")
# Например, если mean_test = 0.46, значит 46% клиентов в тестовой выборке открыли депозит

print(f"\nИнтерпретация:")
print(f"В тестовой выборке {mean_test*100:.1f}% клиентов открыли депозит")
print(f"И {100 - mean_test*100:.1f}% клиентов НЕ открыли депозит")

### Задание 9

In [ ]:
# с помощью SelectKBest отберите 15 наиболее подходящих признаков

# 1. Обрабатываем и X_train, и X_test одинаково
bool_cols_train = X_train.select_dtypes(include=['bool']).columns
bool_cols_test = X_test.select_dtypes(include=['bool']).columns

# Убедимся что у нас одинаковые колонки в train и test
print(f"Bool колонок в X_train: {len(bool_cols_train)}")
print(f"Bool колонок в X_test: {len(bool_cols_test)}")

# 2. Преобразуем bool в int
X_train[bool_cols_train] = X_train[bool_cols_train].astype(int)
X_test[bool_cols_test] = X_test[bool_cols_test].astype(int)

# 3. Убедимся что все колонки одинаковые
missing_in_test = set(X_train.columns) - set(X_test.columns)
missing_in_train = set(X_test.columns) - set(X_train.columns)

if missing_in_test:
    print(f"\nВ X_test отсутствуют колонки: {missing_in_test}")
    # Добавляем недостающие колонки в X_test
    for col in missing_in_test:
        X_test[col] = 0

if missing_in_train:
    print(f"\nВ X_train отсутствуют колонки: {missing_in_train}")
    # Добавляем недостающие колонки в X_train
    for col in missing_in_train:
        X_train[col] = 0

# 4. Упорядочиваем колонки одинаково
X_test = X_test[X_train.columns]

# 5. Теперь SelectKBest
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])

print(f"\nЧисловых колонок в X_train: {X_train_numeric.shape[1]}")
print(f"Числовых колонок в X_test: {X_test_numeric.shape[1]}")

# 6. SelectKBest
selector = SelectKBest(score_func=f_classif, k=15)
X_train_selected = selector.fit_transform(X_train_numeric, y_train)
X_test_selected = selector.transform(X_test_numeric)

# 7. Результаты
selected_features = X_train_numeric.columns[selector.get_support()].tolist()
print(f"\nОтобрано {len(selected_features)} признаков:")

feature_scores = pd.DataFrame({
    'Признак': X_train_numeric.columns,
    'Score': selector.scores_,
    'p-value': selector.pvalues_
}).sort_values('Score', ascending=False)

print("\nТоп-15 признаков:")
for i, (_, row) in enumerate(feature_scores.head(15).iterrows(), 1):
    print(f"{i:2d}. {row['Признак']:30} : Score={row['Score']:.2f}")

print(f"\nРазмер X_train_selected: {X_train_selected.shape}")
print(f"Размер X_test_selected: {X_test_selected.shape}")

### Задание 10

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

# 1. Сохраняем оригинальные данные
X_train_original = X_train_selected.copy()
X_test_original = X_test_selected.copy()

# 2. Создаем MinMaxScaler
scaler = MinMaxScaler()

# 3. Обучаем scaler на тренировочных данных и трансформируем их
X_train_scaled = scaler.fit_transform(X_train_original)  # НОВАЯ переменная!

# 4. Трансформируем тестовые данные
X_test_scaled = scaler.transform(X_test_original)  # НОВАЯ переменная!

print("МАСШТАБИРОВАНИЕ ДАННЫХ (MinMaxScaler):")
print("="*50)
print(f"Размер X_train_scaled: {X_train_scaled.shape}")
print(f"Размер X_test_scaled: {X_test_scaled.shape}")

# 5. Проверяем диапазоны после масштабирования
print("\nДиапазоны значений после масштабирования:")
print(f"X_train_scaled - min: {X_train_scaled.min():.3f}, max: {X_train_scaled.max():.3f}")  # Исправлено!
print(f"X_test_scaled - min: {X_test_scaled.min():.3f}, max: {X_test_scaled.max():.3f}")

# 6. Среднее арифметическое первого предиктора в тестовой выборке
first_predictor_mean = X_test_scaled[:, 0].mean()
first_predictor_mean_rounded = round(first_predictor_mean, 2)

print("\n" + "="*50)
print(f"Первый предиктор (столбец 0) - это: duration")
print(f"Среднее арифметическое первого предиктора в тестовой выборке: {first_predictor_mean:.4f}")
print(f"Среднее арифметическое (округлено до 2 знаков): {first_predictor_mean_rounded}")

# 7. Создаем DataFrame для наглядности (НЕ перезаписывая X_test_scaled!)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=selected_features)

print("\n" + "="*50)
print("СТАТИСТИКИ ДЛЯ ВСЕХ ПРИЗНАКОВ ПОСЛЕ МАСШТАБИРОВАНИЯ:")
print("="*50)

print("\nСредние значения для тестовой выборки:")
for i, (col, mean_val) in enumerate(X_test_scaled_df.mean().items(), 1):
    print(f"{i:2d}. {col:25} : {mean_val:.4f}")

print("\nСтандартные отклонения для тестовой выборки:")
for i, (col, std_val) in enumerate(X_test_scaled_df.std().items(), 1):
    print(f"{i:2d}. {col:25} : {std_val:.4f}")

print(f"\nСреднее арифметическое для первого предиктора (т. е. для первого столбца матрицы) из тестовой выборки: {first_predictor_mean_rounded}") 

# Часть 4: Решение задачи классификации: логистическая регрессия и решающие деревья

### Задание 1

In [ ]:
# обучите логистическую регрессию и рассчитайте метрики качества

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Модель логистической регрессии
log_reg_model = LogisticRegression(solver='sag', max_iter=1000, random_state=42)
log_reg_model.fit(X_train_scaled, y_train)

# Предсказываем на масштабированных тестовых данных
y_pred = log_reg_model.predict(X_test_scaled)

# Оценка
accuracy = round(accuracy_score(y_test, y_pred), 2)
print(f"Accuracy: {accuracy}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred)) 

### Задания 2, 3, 4

In [ ]:
# обучите решающие деревья, настройте максимальную глубину

from sklearn.tree import DecisionTreeClassifier

# 1. Базовое решающее дерево
dt_model = DecisionTreeClassifier(criterion='entropy', random_state=42)
dt_model.fit(X_train_scaled, y_train)

# 2. Предсказания и оценка
y_pred_dt = dt_model.predict(X_test_scaled)
accuracy_dt = accuracy_score(y_test, y_pred_dt)

print("РЕШАЮЩЕЕ ДЕРЕВО (базовая модель):")
print("="*50)
print(f"Accuracy на тестовой выборке: {accuracy_dt:.4f}")
print(f"Accuracy (округлено до 2 знаков): {round(accuracy_dt, 2)}")
print(f"\nГлубина дерева: {dt_model.get_depth()}")
print(f"Количество листьев: {dt_model.get_n_leaves()}")

# Перебираем глубины деревьев
max_depths = range(1, 31)  # от 1 до 30
train_accuracies = []
test_accuracies = []

print("ПОИСК ОПТИМАЛЬНОЙ ГЛУБИНЫ ДЕРЕВА:")
print("="*50)

best_depth = 1
best_accuracy = 0

for depth in max_depths:
    # Обучаем дерево с заданной глубиной
    dt = DecisionTreeClassifier(criterion='entropy', max_depth=depth, random_state=42)
    dt.fit(X_train_scaled, y_train)
    
    # Accuracy на обучающей и тестовой выборках
    train_acc = dt.score(X_train_scaled, y_train)
    test_acc = dt.score(X_test_scaled, y_test)
    
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)
    
    # Ищем лучшую accuracy на тесте
    if test_acc > best_accuracy:
        best_accuracy = test_acc
        best_depth = depth
    
    print(f"Глубина {depth:2d}: train={train_acc:.4f}, test={test_acc:.4f}")

print("\n" + "="*50)
print(f"ОПТИМАЛЬНАЯ ГЛУБИНА: {best_depth}")
print(f"Лучшая accuracy на тесте: {best_accuracy:.4f}")
print(f"Лучшая accuracy (округлено до 2 знаков): {round(best_accuracy, 2)}")

print(f"Наибольшее значение accuracy: {round(best_accuracy, 2)}")
print(f"Оптимальная глубина дерева: {best_depth}") 

### Задание 5

In [ ]:
# подберите оптимальные параметры с помощью gridsearch

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

# 1. Определяем параметры для GridSearch
param_grid = {
    'min_samples_split': [2, 5, 7, 10],
    'max_depth': [3, 5, 7]
}

# 2. Создаем модель и GridSearch
dt = DecisionTreeClassifier(criterion='entropy', random_state=42)
grid_search = GridSearchCV(
    estimator=dt,
    param_grid=param_grid,
    scoring='f1',  # оптимизируем по F1-score
    cv=5,  # 5-кратная кросс-валидация
    n_jobs=-1,  # использовать все ядра процессора
    verbose=1
)

# 3. Обучаем GridSearch
print("ЗАПУСК GRIDSEARCH...")
grid_search.fit(X_train_scaled, y_train)

# 4. Результаты
print("\n" + "="*50)
print("РЕЗУЛЬТАТЫ GRIDSEARCH:")
print("="*50)
print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"Лучший F1-score (кросс-валидация): {grid_search.best_score_:.4f}")

# 5. Оценка на тестовой выборке
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test_scaled)

# Вычисляем метрики
f1_test = f1_score(y_test, y_pred_best)
accuracy_test = accuracy_score(y_test, y_pred_best)

print(f"\nМЕТРИКИ НА ТЕСТОВОЙ ВЫБОРКЕ:")
print(f"F1-score: {f1_test:.4f}")
print(f"F1-score (округлено до 2 знаков): {round(f1_test, 2)}")
print(f"Accuracy: {accuracy_test:.4f}")

# 6. Подробная информация по всем комбинациям
print("\n" + "="*50)
print("ВСЕ КОМБИНАЦИИ ПАРАМЕТРОВ:")
print("="*50)

results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']]
results_df = results_df.sort_values('rank_test_score')

for i, row in results_df.iterrows():
    params = row['params']
    score = row['mean_test_score']
    std = row['std_test_score']
    rank = row['rank_test_score']
    print(f"Ранг {rank}: params={params}, F1={score:.4f} (±{std:.4f})")

print(f"\nМетрика F1 на тестовой выборке для наилучшей комбинации перебираемых параметров: {round(f1_test, 2)}")

# Часть 5: Решение задачи классификации: ансамбли моделей и построение прогноза

### Задание 1

In [ ]:
# обучите на ваших данных случайный лес

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score

# 1. Обучаем случайный лес
rf_model = RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    min_samples_leaf=5,
    max_depth=10,
    random_state=42,
    n_jobs=-1  # используем все ядра процессора
)

print("ОБУЧЕНИЕ СЛУЧАЙНОГО ЛЕСА...")
rf_model.fit(X_train_scaled, y_train)

# 2. Предсказания
y_pred_rf = rf_model.predict(X_test_scaled)
y_pred_proba_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

# 3. Вычисляем метрики
accuracy_rf = accuracy_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)  # recall для класса 1 (deposit=yes)

print("\n" + "="*50)
print("РЕЗУЛЬТАТЫ СЛУЧАЙНОГО ЛЕСА:")
print("="*50)
print(f"Accuracy: {accuracy_rf:.4f}")
print(f"Accuracy (округлено до 2 знаков): {round(accuracy_rf, 2)}")
print(f"Recall: {recall_rf:.4f}")
print(f"Recall (округлено до 2 знаков): {round(recall_rf, 2)}")

# 4. Дополнительные метрики
from sklearn.metrics import classification_report, confusion_matrix

print(f"\nCLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred_rf))

# 5. Важность признаков
print("ВАЖНОСТЬ ПРИЗНАКОВ (feature importances):")
feature_importance = pd.DataFrame({
    'Признак': selected_features,
    'Важность': rf_model.feature_importances_
}).sort_values('Важность', ascending=False)

for i, (_, row) in enumerate(feature_importance.iterrows(), 1):
    print(f"{i:2d}. {row['Признак']:25} : {row['Важность']:.4f}")

# 6. Визуализация важности признаков
plt.figure(figsize=(12, 8))
feature_importance_sorted = feature_importance.sort_values('Важность')
plt.barh(feature_importance_sorted['Признак'], feature_importance_sorted['Важность'])
plt.title('Важность признаков в случайном лесе')
plt.xlabel('Важность')
plt.ylabel('Признак')
plt.tight_layout()
plt.show()

print(f"\n" + "="*50)
print("Метрики для построенной модели на тестовой выборке:")
print(f"Accuracy: {round(accuracy_rf, 2)}")
print(f"Recall: {round(recall_rf, 2)}")

### Задания 2 и 3

In [ ]:
# используйте для классификации градиентный бустинг и сравните качество со случайным лесом

from sklearn.ensemble import GradientBoostingClassifier

# 1. Обучаем градиентный бустинг
gb_model = GradientBoostingClassifier(
    learning_rate=0.05,
    n_estimators=300,
    min_samples_leaf=5,
    max_depth=5,
    random_state=42
)

print("ОБУЧЕНИЕ ГРАДИЕНТНОГО БУСТИНГА...")
gb_model.fit(X_train_scaled, y_train)

# 2. Предсказания
y_pred_gb = gb_model.predict(X_test_scaled)
y_pred_proba_gb = gb_model.predict_proba(X_test_scaled)[:, 1]

# 3. Вычисляем метрики
f1_gb = f1_score(y_test, y_pred_gb)
accuracy_gb = accuracy_score(y_test, y_pred_gb)

print("\n" + "="*50)
print("РЕЗУЛЬТАТЫ ГРАДИЕНТНОГО БУСТИНГА:")
print("="*50)
print(f"F1-score: {f1_gb:.4f}")
print(f"F1-score (округлено до 2 знаков): {round(f1_gb, 2)}")
print(f"Accuracy: {accuracy_gb:.4f}")
print(f"Accuracy (округлено до 2 знаков): {round(accuracy_gb, 2)}")

# 4. Дополнительные метрики
print(f"\nCLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred_gb))

print("="*50)
print(f"F1 на тестовой выборке: {round(f1_gb, 2)}")

### Задание 4

In [ ]:
# объедините уже известные вам алгоритмы с помощью стекинга 

from sklearn.ensemble import StackingClassifier
from sklearn.metrics import precision_score

# 1. Определяем базовые модели (те же параметры что и раньше)
base_models = [
    ('decision_tree', DecisionTreeClassifier(
        criterion='entropy', 
        max_depth=7,  # оптимальная глубина 
        random_state=42
    )),
    ('logistic_regression', LogisticRegression(
        solver='sag',
        max_iter=1000,
        random_state=42
    )),
    ('gradient_boosting', GradientBoostingClassifier(
        learning_rate=0.05,
        n_estimators=300,
        min_samples_leaf=5,
        max_depth=5,
        random_state=42
    ))
]

# 2. Создаем стекинг с логистической регрессией как мета-моделью
stacking_model = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(
        solver='sag',
        max_iter=1000,
        random_state=42
    ),
    cv=5,  # 5-кратная кросс-валидация для обучения мета-модели
    n_jobs=-1
)

print("ОБУЧЕНИЕ СТЕКИНГ-МОДЕЛИ...")
stacking_model.fit(X_train_scaled, y_train)

# 3. Предсказания
y_pred_stacking = stacking_model.predict(X_test_scaled)
y_pred_proba_stacking = stacking_model.predict_proba(X_test_scaled)[:, 1]

# 4. Вычисляем метрики
precision_stacking = precision_score(y_test, y_pred_stacking)
accuracy_stacking = accuracy_score(y_test, y_pred_stacking)
recall_stacking = recall_score(y_test, y_pred_stacking)
f1_stacking = f1_score(y_test, y_pred_stacking)

print("\n" + "="*50)
print("РЕЗУЛЬТАТЫ СТЕКИНГ-МОДЕЛИ:")
print("="*50)
print(f"Precision: {precision_stacking:.4f}")
print(f"Precision (округлено до 2 знаков): {round(precision_stacking, 2)}")
print(f"Accuracy: {accuracy_stacking:.4f}")
print(f"Recall: {recall_stacking:.4f}")
print(f"F1-score: {f1_stacking:.4f}")

# 5. Сравнение всех моделей
print("\n" + "="*50)
print("СРАВНЕНИЕ ВСЕХ МОДЕЛЕЙ:")
print("="*50)

comparison = pd.DataFrame({
    'Модель': ['Логистическая регрессия', 'Решающее дерево', 'Random Forest', 
               'Gradient Boosting', 'Stacking (ансамбль)'],
    'Accuracy': [0.80, 0.80, 0.83, 0.82, round(accuracy_stacking, 2)],
    'Precision': [
        round(precision_score(y_test, log_reg_model.predict(X_test_scaled)), 2),
        round(precision_score(y_test, DecisionTreeClassifier(criterion='entropy', max_depth=7, random_state=42)
                              .fit(X_train_scaled, y_train).predict(X_test_scaled)), 2),
        0.81,  # из classification report Random Forest
        0.80,  # из classification report Gradient Boosting  
        round(precision_stacking, 2)
    ],
    'Recall': [0.73, 0.73, 0.83, 0.83, round(recall_stacking, 2)],
    'F1-score': [0.78, 0.78, 0.82, 0.81, round(f1_stacking, 2)]
})

print(comparison.to_string(index=False))

# 6. Classification report для стекинга
print(f"\nCLASSIFICATION REPORT (Stacking):")
print(classification_report(y_test, y_pred_stacking))

print("="*50)
print(f"Метрика precision на тестовой выборке: {round(precision_stacking, 2)}")

### Задание 5

In [ ]:
# оцените, какие признаки демонстрируют наибольшую  важность в модели градиентного бустинга

# 1. Важность признаков
print("ВАЖНОСТЬ ПРИЗНАКОВ (feature importances):")
feature_importance_gb = pd.DataFrame({
    'Признак': selected_features,
    'Важность': gb_model.feature_importances_
}).sort_values('Важность', ascending=False)

for i, (_, row) in enumerate(feature_importance_gb.iterrows(), 1):
    print(f"{i:2d}. {row['Признак']:25} : {row['Важность']:.4f}")

# 2. Сравнение с другими моделями
print("\n" + "="*50)
print("СРАВНЕНИЕ МОДЕЛЕЙ:")
print("="*50)

models_comparison = pd.DataFrame({
    'Модель': ['Логистическая регрессия', 'Решающее дерево', 'Random Forest', 'Gradient Boosting'],
    'Accuracy': [0.80, 0.80, 0.83, round(accuracy_gb, 2)],
    'F1-score': [0.78, 0.78, 0.82, round(f1_gb, 2)],
    'Recall': [0.73, 0.73, 0.83, round(recall_score(y_test, y_pred_gb), 2)]
})

print(models_comparison.to_string(index=False))

# 3. Визуализация важности признаков
plt.figure(figsize=(12, 8))
feature_importance_gb_sorted = feature_importance_gb.sort_values('Важность')
plt.barh(feature_importance_gb_sorted['Признак'], feature_importance_gb_sorted['Важность'])
plt.title('Важность признаков в градиентном бустинге')
plt.xlabel('Важность')
plt.ylabel('Признак')
plt.tight_layout()
plt.show()

### Задания 6, 7, 8

In [ ]:
# реализуйте оптимизацию гиперпараметров с помощью Optuna

import optuna
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import f1_score, accuracy_score

# 1. Определяем целевую функцию для Optuna
def objective(trial):
    # Параметры для оптимизации
    n_estimators = trial.suggest_int('n_estimators', 100, 200, 1)
    max_depth = trial.suggest_int('max_depth', 10, 30, 1)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 2, 10, 1)
    
    # Создаем и обучаем модель
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        criterion='gini',
        random_state=42,
        n_jobs=-1
    )
    
    # Используем кросс-валидацию для оценки
    from sklearn.model_selection import cross_val_score
    scores = cross_val_score(model, X_train_scaled, y_train, 
                            cv=3, scoring='f1', n_jobs=-1)
    
    # Возвращаем средний F1-score (Optuna максимизирует)
    return scores.mean()

# 2. Создаем study и оптимизируем
print("ЗАПУСК OPTUNA ДЛЯ ОПТИМИЗАЦИИ RANDOM FOREST...")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)

# 3. Результаты оптимизации
print("\n" + "="*50)
print("РЕЗУЛЬТАТЫ OPTUNA ОПТИМИЗАЦИИ:")
print("="*50)
print(f"Лучшее значение F1 (кросс-валидация): {study.best_value:.4f}")
print(f"Лучшие параметры: {study.best_params}")
print(f"Количество trials: {len(study.trials)}")

# 4. Обучаем финальную модель с лучшими параметрами
best_params = study.best_params
best_rf = RandomForestClassifier(
    n_estimators=best_params['n_estimators'],
    max_depth=best_params['max_depth'],
    min_samples_leaf=best_params['min_samples_leaf'],
    criterion='gini',
    random_state=42,
    n_jobs=-1
)

best_rf.fit(X_train_scaled, y_train)

# 5. Оценка на тестовой выборке
y_pred_optuna = best_rf.predict(X_test_scaled)
f1_optuna = f1_score(y_test, y_pred_optuna)
accuracy_optuna = accuracy_score(y_test, y_pred_optuna)

print("\n" + "="*50)
print("РЕЗУЛЬТАТЫ НА ТЕСТОВОЙ ВЫБОРКЕ:")
print("="*50)
print(f"F1-score: {f1_optuna:.4f}")
print(f"F1-score (округлено до 2 знаков): {round(f1_optuna, 2)}")
print(f"Accuracy: {accuracy_optuna:.4f}")
print(f"Accuracy (округлено до 2 знаков): {round(accuracy_optuna, 2)}")

# 6. Сравнение с исходным Random Forest
print("\n" + "="*50)
print("СРАВНЕНИЕ С ИСХОДНЫМ RANDOM FOREST:")
print("="*50)
print(f"Исходный RF (n_estimators=100, max_depth=10, min_samples_leaf=5):")
print(f"  F1: 0.82, Accuracy: 0.83")
print(f"\nOptuna оптимизированный RF ({best_params}):")
print(f"  F1: {round(f1_optuna, 2)}, Accuracy: {round(accuracy_optuna, 2)}")

# 7. Визуализация результатов Optuna
try:
    fig1 = optuna.visualization.plot_optimization_history(study)
    fig1.show()
    
    fig2 = optuna.visualization.plot_param_importances(study)
    fig2.show()
    
    fig3 = optuna.visualization.plot_slice(study)
    fig3.show()
except:
    print("\nДля визуализации Optuna установите: pip install plotly")

print("="*50)
print(f"F1-score на тестовой выборке: {round(f1_optuna, 2)}")
print(f"Accuracy на тестовой выборке: {round(accuracy_optuna, 2)}")